# Scaling laws and emergent abilities

Scaling laws say loss falls smoothly with model size and data, even when abilities appear suddenly.

Power-law curves connect parameters, data, and loss. Thresholded benchmarks can look abrupt even when the underlying loss curve is smooth, so validation under shift matters. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(95)


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-float(x)))


def scaling_loss(N, D, quality=1.0, L_inf=1.0, a=2.0, b=3.0, alpha=0.5, beta=0.5):
    model_term = a * N ** (-alpha)
    data_term = b * D ** (-beta)
    quality_penalty = 0.18 * (1.0 / quality - 1.0)
    total = L_inf + model_term + data_term + quality_penalty
    return total, model_term, data_term, quality_penalty


def scaling_ladder():
    return [
        {"name": "D1 one compute point", "N": np.array([100.0]), "D": np.array([100.0]), "quality": 1.0},
        {"name": "D2 clean rising N", "N": np.array([100.0, 200.0, 400.0, 800.0]), "D": np.array([900.0, 900.0, 900.0, 900.0]), "quality": 1.0},
        {"name": "D3 noisy loss measurements", "N": np.array([80.0, 160.0, 320.0, 640.0]), "D": np.array([700.0, 800.0, 900.0, 1000.0]), "quality": 0.95},
        {"name": "D4 fitted synthetic experiments", "N": np.array([120.0, 240.0, 480.0, 960.0]), "D": np.array([400.0, 700.0, 1000.0, 1400.0]), "quality": 1.0},
        {"name": "D5 shifted data quality", "N": np.array([200.0, 400.0, 800.0, 1600.0]), "D": np.array([500.0, 900.0, 1400.0, 2000.0]), "quality": 0.65},
    ]

## The concept, built once: scaling loss and thresholded ability

A small power law writes loss as
$$L(N,D)=L_\infty+aN^{-\alpha}+bD^{-\beta}.$$
The lesson uses $L_\infty=1$, $a=2$, $b=3$, $\alpha=\beta=0.5$ to separate model-size and data terms.

In [ ]:
def ability_score(loss, threshold=2.7, sharpness=2.0):
    margin = sharpness * (threshold - loss)
    return sigmoid(margin)

The exact worked checks are model term $2/\sqrt{100}=0.200$, increasing $N$ to 400 gives $0.100$, data term $3/\sqrt{900}=0.100$, total loss at $N=400,D=900$ is 1.200, and sigmoid margin 1.5 gives 0.818.

In [ ]:
_, model_100, _, _ = scaling_loss(100.0, 900.0)
_, model_400, data_900, _ = scaling_loss(400.0, 900.0)
total_400_900, _, _, _ = scaling_loss(400.0, 900.0)
ability = sigmoid(1.5)

assert round(model_100, 3) == 0.200
assert round(model_400, 3) == 0.100
assert round(data_900, 3) == 0.100
assert round(total_400_900, 3) == 1.200
assert round(ability, 3) == 0.818

print("model term N=100", round(model_100, 3))
print("model term N=400", round(model_400, 3))
print("data term D=900", round(data_900, 3))
print("total loss", round(total_400_900, 3))
print("ability score", round(ability, 3))

## The dataset ladder

D1-D5 builds a compute/parameter/data ladder, then adds noisy observations and a data-quality shift where naive extrapolation breaks.

In [ ]:
ladder = scaling_ladder()
for rung in ladder:
    compute = rung["N"] * rung["D"]
    print(rung["name"], "points", len(rung["N"]), "compute range", (float(compute.min()), float(compute.max())), "quality", rung["quality"])

In [ ]:
rows = []
for rung in ladder:
    predicted = []
    observed = []
    abilities = []
    for N_value, D_value in zip(rung["N"], rung["D"]):
        loss, _, _, _ = scaling_loss(float(N_value), float(D_value))
        shifted, _, _, penalty = scaling_loss(float(N_value), float(D_value), quality=rung["quality"])
        noise = float(RNG.normal(0.0, 0.015)) if len(rung["N"]) > 1 else 0.0
        predicted.append(loss)
        observed.append(shifted + noise)
        abilities.append(ability_score(shifted))
    prediction_error = float(np.mean(np.abs(np.array(predicted) - np.array(observed))))
    rows.append({
        "name": rung["name"],
        "compute": rung["N"] * rung["D"],
        "predicted": np.array(predicted),
        "observed": np.array(observed),
        "error": prediction_error,
        "ability": float(np.mean(abilities)),
    })
for row in rows:
    print(f"{row['name']}: MAE={row['error']:.3f} ability={row['ability']:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, row in enumerate(rows):
    axes[0, idx].plot(row["compute"], row["predicted"], marker="o", label="power law")
    axes[0, idx].plot(row["compute"], row["observed"], marker="x", label="observed")
    axes[0, idx].set_xscale("log")
    axes[0, idx].set_title(f"D{idx + 1}")
    axes[1, idx].bar(["MAE"], [row["error"]], color="slateblue")
axes[0, 0].legend()
fig.suptitle("Loss-vs-compute panels and prediction error")
plt.figure(figsize=(6, 3))
plt.plot([float(np.mean(row["compute"])) for row in rows], [row["error"] for row in rows], marker="o")
plt.xscale("log")
plt.xlabel("mean compute N times D")
plt.ylabel("loss prediction error")
plt.grid(True)

## Pitfall on D5: treating a power law as destiny

A fitted power law can be numerically accurate in-distribution and fail when data quality shifts. The fix is to hold out shifted validation and add a data-quality term.

In [ ]:
hard = ladder[-1]
naive = []
quality_aware = []
observed = []
for N_value, D_value in zip(hard["N"], hard["D"]):
    base, _, _, _ = scaling_loss(float(N_value), float(D_value), quality=1.0)
    shifted, _, _, _ = scaling_loss(float(N_value), float(D_value), quality=hard["quality"])
    corrected, _, _, _ = scaling_loss(float(N_value), float(D_value), quality=hard["quality"])
    naive.append(base)
    observed.append(shifted)
    quality_aware.append(corrected)
naive_error = float(np.mean(np.abs(np.array(naive) - np.array(observed))))
fixed_error = float(np.mean(np.abs(np.array(quality_aware) - np.array(observed))))
print("naive extrapolation error", round(naive_error, 3))
print("quality-aware validation error", round(fixed_error, 3))

## Evaluate it + practice

            - Metric: loss prediction error or ability accuracy; compare to the no-skill baseline, predict the mean observed loss.
            - Cheap sanity check: doubling model size reduces the model term by the expected power.
            - Ablation: remove the data-quality term and evaluate D5.
            - Failure signals: validation loss shifts upward while in-distribution curves still look smooth.
            - Practice: Fit an illustrative exponent from D2 points.
- Practice: Move the ability threshold and plot the sigmoid curve.
- Practice: Compare compute increases from scaling N versus D.